# 02_clean.ipynb - 数据清洗

本 Notebook 负责数据清洗和存储：
- 单表清洗（缺失值、日期格式、数据类型、重复值、离群值）
- 宽表与长表转换
- 多表合并
- CSV 与 Parquet 存储对比

In [1]:
import os
import sys
import time
import pandas as pd
import numpy as np
import pyarrow.parquet as pq

# 项目路径设置 - 直接使用当前工作目录
project_root = os.getcwd()
sys.path.insert(0, project_root)
print(f"Project root: {project_root}")

Project root: c:\Users\86155\Desktop\新建文件夹\dshw-p01


In [2]:
# 配置信息
STOCKS = [
    ("601398", "工商银行", "银行"),
    ("000001", "平安银行", "银行"),
    ("600104", "上汽集团", "汽车"),
    ("002594", "比亚迪", "汽车"),
    ("000002", "万科A", "房地产"),
    ("600048", "保利发展", "房地产"),
    ("600519", "贵州茅台", "白酒"),
    ("000858", "五粮液", "白酒"),
    ("601088", "中国神华", "能源"),
    ("600050", "中国联通", "通讯")
]

INDICES = [
    ("000300", "沪深300"),
    ("000905", "中证500")
]

## 3.1 单表清洗

In [3]:
def clean_single_stock(code, name):
    """清洗单只股票数据"""
    input_path = os.path.join(project_root, "data", "stock", f"stock_{code}.csv")
    df = pd.read_csv(input_path)
    
    print(f"\n========== {code} {name} 清洗前 ==========")
    print(f"原始形状: {df.shape}")
    
    # 1. 缺失值检测
    missing_stats = pd.DataFrame({
        '缺失数量': df.isnull().sum(),
        '缺失比例': df.isnull().sum() / len(df) * 100
    })
    print("\n缺失值统计:")
    print(missing_stats)
    
    # 2. 缺失值处理：向前填充
    df_clean = df.ffill().copy()
    remaining_missing = df_clean.isnull().sum().sum()
    if remaining_missing > 0:
        df_clean = df_clean.dropna()
        print(f"\n删除了 {remaining_missing} 个仍存在的缺失值")
    
    # 3. 日期格式统一
    df_clean['date'] = pd.to_datetime(df_clean['date'])
    df_clean = df_clean.set_index('date')
    
    # 4. 数据类型检查
    numeric_cols = ['open', 'close', 'high', 'low', 'volume', 'amount']
    for col in numeric_cols:
        if df_clean[col].dtype == 'object':
            df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
            print(f"\n将 {col} 转换为数值型")
    
    # 5. 重复值处理
    duplicate_count = df_clean.duplicated().sum()
    if duplicate_count > 0:
        df_clean = df_clean.drop_duplicates()
        print(f"\n删除了 {duplicate_count} 行重复值")
    
    # 6. 离群值标注
    df_clean['return'] = np.log(df_clean['close'] / df_clean['close'].shift(1))
    df_clean['is_extreme'] = abs(df_clean['return']) > 0.2
    extreme_count = df_clean['is_extreme'].sum()
    print(f"\n标注了 {extreme_count} 个极端涨跌交易日 (|涨跌幅| > 20%)")
    
    print(f"\n清洗后形状: {df_clean.shape}")
    
    return df_clean.reset_index()

In [4]:
# 清洗财务数据（转换为小数形式 + 补齐股票代码）
finance_input_path = os.path.join(project_root, "data", "finance", "finance_ratios.csv")
finance_df = pd.read_csv(finance_input_path, dtype={'code': str})

print(f"\n========== 财务数据清洗 ==========")
print(f"原始形状: {finance_df.shape}")
print("\n原始数据前5行:")
print(finance_df.head())
print(f"\n原始code类型: {finance_df['code'].dtype}")

# 强制补齐股票代码为6位
def pad_code(x):
    s = str(x)
    return s.zfill(6)

finance_df['code'] = finance_df['code'].apply(pad_code)

# 将百分比转换为小数形式（除以100）
finance_df['value'] = finance_df['value'] / 100

print("\n处理后数据前5行:")
print(finance_df.head())
print(f"\n处理后code类型: {finance_df['code'].dtype}")

# 保存清洗后的财务数据
finance_output_path = os.path.join(project_root, "data", "clean", "finance_clean.csv")
finance_df.to_csv(finance_output_path, index=False, encoding="utf-8-sig")
print(f"\n清洗后的财务数据已保存至: {finance_output_path}")


========== 财务数据清洗 ==========
原始形状: (100, 4)

原始数据前5行:
     code  year          indicator    value
0  601398  2020                ROE  11.3438
1  601398  2020  net_profit_margin  35.9916
2  601398  2021                ROE  11.3258
3  601398  2021  net_profit_margin  37.1479
4  601398  2022                ROE  10.6764

原始code类型: object

处理后数据前5行:
     code  year          indicator     value
0  601398  2020                ROE  0.113438
1  601398  2020  net_profit_margin  0.359916
2  601398  2021                ROE  0.113258
3  601398  2021  net_profit_margin  0.371479
4  601398  2022                ROE  0.106764

处理后code类型: object

清洗后的财务数据已保存至: c:\Users\86155\Desktop\新建文件夹\dshw-p01\data\clean\finance_clean.csv


In [5]:
# 清洗所有股票并合并
all_stocks_clean = []

for code, name, industry in STOCKS:
    try:
        df_clean = clean_single_stock(code, name)
        df_clean['code'] = str(code).zfill(6)
        df_clean['name'] = name
        df_clean['industry'] = industry
        all_stocks_clean.append(df_clean)
    except Exception as e:
        print(f"清洗 {code} {name} 失败: {e}")

# 合并所有股票清洗后的数据
stock_clean_df = pd.concat(all_stocks_clean, ignore_index=True)

# 保存清洗后的股票数据 (CSV)
csv_output = os.path.join(project_root, "data", "clean", "stock_clean.csv")
stock_clean_df.to_csv(csv_output, index=False, encoding="utf-8-sig")
print(f"\n清洗后的股票数据已保存至: {csv_output}")


========== 601398 工商银行 清洗前 ==========
原始形状: (1516, 7)

缺失值统计:
        缺失数量  缺失比例
date       0   0.0
open       0   0.0
close      0   0.0
high       0   0.0
low        0   0.0
volume     0   0.0
amount     0   0.0

标注了 0 个极端涨跌交易日 (|涨跌幅| > 20%)

清洗后形状: (1516, 8)

========== 000001 平安银行 清洗前 ==========
原始形状: (1515, 7)

缺失值统计:
        缺失数量  缺失比例
date       0   0.0
open       0   0.0
close      0   0.0
high       0   0.0
low        0   0.0
volume     0   0.0
amount     0   0.0

标注了 0 个极端涨跌交易日 (|涨跌幅| > 20%)

清洗后形状: (1515, 8)

========== 600104 上汽集团 清洗前 ==========
原始形状: (1515, 7)

缺失值统计:
        缺失数量  缺失比例
date       0   0.0
open       0   0.0
close      0   0.0
high       0   0.0
low        0   0.0
volume     0   0.0
amount     0   0.0

标注了 0 个极端涨跌交易日 (|涨跌幅| > 20%)

清洗后形状: (1515, 8)

========== 002594 比亚迪 清洗前 ==========
原始形状: (1515, 7)

缺失值统计:
        缺失数量  缺失比例
date       0   0.0
open       0   0.0
close      0   0.0
high       0   0.0
low        0   0.0
volume     0   0.0
amount     0   0

### 缺失值说明

缺失值可能的原因：
1. **停牌**：股票因重大事项停牌期间没有交易数据
2. **节假日/非交易日**：A股市场在周末和法定节假日休市
3. **数据源问题**：数据供应商可能存在部分数据缺失

选择向前填充(ffill)的原因：股票价格具有连续性，前一个交易日的价格是对缺失值的合理估计。

### 极端值说明

单日涨跌幅超过 ±20% 的可能原因：
1. **新股上市**：上市首日不受涨跌幅限制
2. **重组/复牌**：重大资产重组后复牌首日可能无涨跌幅限制
3. **分红除权**：虽然数据已做复权处理，但极端情况下仍可能有大幅波动
4. **数据错误**：数据源可能存在异常值

## 3.2 宽表与长表转换

In [6]:
# 构建收盘价宽表
close_data = []
for code, name, industry in STOCKS:
    input_path = os.path.join(project_root, "data", "stock", f"stock_{code}.csv")
    df = pd.read_csv(input_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    close_series = df['close'].rename(code)
    close_data.append(close_series)

# 宽表：日期为索引，每列一只股票
close_wide = pd.concat(close_data, axis=1)
print("收盘价宽表 (前5行):")
print(close_wide.head())
print(f"\n宽表形状: {close_wide.shape}")

收盘价宽表 (前5行):
              601398     000001     600104     002594     000002     600048  \
date                                                                          
2020-01-02  4.248007  13.684725  20.212888  15.582369  26.594324  12.602703   
2020-01-03  4.262238  13.936193  20.154348  15.540315  26.177767  12.362430   
2020-01-06  4.248007  13.846962  19.535501  15.617952  25.736706  12.153160   
2020-01-07  4.276470  13.911857  19.677668  15.543550  25.940901  12.238418   
2020-01-08  4.205314  13.514375  19.978729  15.294465  25.875559  12.029148   

                600519      000858     601088    600050  
date                                                     
2020-01-02  979.045560  111.862382  11.610515  5.153105  
2020-01-03  934.477327  110.566581  11.591778  5.153105  
2020-01-06  933.983472  109.423227  11.729180  5.136041  
2020-01-07  948.313926  109.567205  11.754163  5.136041  
2020-01-08  942.777554  109.160679  11.460621  5.050725  

宽表形状: (1516, 10)


In [7]:
# 转换为长表
close_long = close_wide.reset_index().melt(
    id_vars=['date'],
    var_name='code',
    value_name='close'
)
print("收盘价长表 (前5行):")
print(close_long.head())
print(f"\n长表形状: {close_long.shape}")

收盘价长表 (前5行):
        date    code     close
0 2020-01-02  601398  4.248007
1 2020-01-03  601398  4.262238
2 2020-01-06  601398  4.248007
3 2020-01-07  601398  4.276470
4 2020-01-08  601398  4.205314

长表形状: (15160, 3)


### 宽表 vs 长表

**宽表适合的操作**：
- 计算多只股票之间的相关系数矩阵
- 同时查看多只股票在同一时间点的表现
- 进行矩阵运算（如投资组合优化）
- 绘制多只股票的对比走势图

**长表适合的操作**：
- 按股票分组统计分析
- 使用 seaborn 等绘图库进行分面可视化
- 与其他表进行 merge 操作
- 存储标准化的时间序列数据

## 3.3 多表合并

In [8]:
# 1. 加载指数数据
index_data = {}
for code, name in INDICES:
    input_path = os.path.join(project_root, "data", "index", f"index_{code}.csv")
    df = pd.read_csv(input_path)
    df['date'] = pd.to_datetime(df['date'])
    df = df.set_index('date')
    df = df.add_prefix(f'index_{code}_')
    index_data[code] = df

# 合并指数数据
index_combined = pd.concat(index_data.values(), axis=1).reset_index()
print(f"指数数据形状: {index_combined.shape}")

指数数据形状: (1516, 13)


In [9]:
# 2. 将股票数据与指数数据按日期 left join
print(f"合并前股票数据行数: {len(stock_clean_df)}")

combined_df = pd.merge(
    stock_clean_df,
    index_combined,
    on='date',
    how='left'
)

print(f"与指数合并后行数: {len(combined_df)}")
print(f"行数变化: {len(combined_df) - len(stock_clean_df)}")

合并前股票数据行数: 15142
与指数合并后行数: 15142
行数变化: 0


In [10]:
# 3. 加载并合并宏观数据
cpi_path = os.path.join(project_root, "data", "macro", "macro_cpi.csv")
fx_path = os.path.join(project_root, "data", "macro", "macro_fx.csv")

cpi_df = pd.read_csv(cpi_path)
fx_df = pd.read_csv(fx_path)

# 合并宏观数据
macro_df = pd.merge(cpi_df, fx_df, on='date', how='outer')
macro_df = macro_df.rename(columns={'date': 'month'})

# 为股票数据添加月份列，用于合并
combined_df['month'] = combined_df['date'].dt.strftime('%Y-%m')

print(f"与宏观数据合并前行数: {len(combined_df)}")

# 合并宏观数据
combined_df = pd.merge(
    combined_df,
    macro_df,
    on='month',
    how='left'
)

print(f"与宏观数据合并后行数: {len(combined_df)}")
print(f"行数变化: {len(combined_df) - len(combined_df)}")

与宏观数据合并前行数: 15142
与宏观数据合并后行数: 15142
行数变化: 0


### 合并说明

1. **与指数数据合并**：使用 left join，以股票交易日为基准，确保所有股票交易日都保留。行数不变是因为指数数据在交易日也是齐全的。

2. **与宏观数据合并**：宏观数据是月度频率，股票数据是日度频率。通过将股票数据按月份分组，将月度宏观数据映射到该月的每个交易日。

In [11]:
# 保存合并后的综合数据
combined_output = os.path.join(project_root, "data", "combined", "combined_data.csv")
combined_df.to_csv(combined_output, index=False, encoding="utf-8-sig")
print(f"合并后的综合数据已保存至: {combined_output}")

合并后的综合数据已保存至: c:\Users\86155\Desktop\新建文件夹\dshw-p01\data\combined\combined_data.csv


## 方式 B：Parquet 存储与 CSV 对比

In [12]:
# 同时保存 Parquet 格式
parquet_output = os.path.join(project_root, "data", "clean", "stock_clean.parquet")
stock_clean_df.to_parquet(parquet_output, index=False)
print(f"清洗后的股票数据 (Parquet) 已保存至: {parquet_output}")

清洗后的股票数据 (Parquet) 已保存至: c:\Users\86155\Desktop\新建文件夹\dshw-p01\data\clean\stock_clean.parquet


In [13]:
# 1. 列式读取演示
print("=== 列式读取演示 ===")
df_parquet = pd.read_parquet(parquet_output, columns=["date", "code", "close"])
print("只读取 date, code, close 三列:")
print(df_parquet.head())

=== 列式读取演示 ===
只读取 date, code, close 三列:
        date    code     close
0 2020-01-02  601398  4.248007
1 2020-01-03  601398  4.262238
2 2020-01-06  601398  4.248007
3 2020-01-07  601398  4.276470
4 2020-01-08  601398  4.205314


In [14]:
# 2. 查看 Schema
print("\n=== Parquet Schema ===")
schema = pq.read_schema(parquet_output)
print(schema)


=== Parquet Schema ===
date: timestamp[ns]
open: double
close: double
high: double
low: double
volume: double
amount: double
return: double
is_extreme: bool
code: string
name: string
industry: string
-- schema metadata --
pandas: '{"index_columns": [], "column_indexes": [], "columns": [{"name":' + 1432


In [15]:
# 3. 读取速度与文件体积对比
csv_path = os.path.join(project_root, "data", "clean", "stock_clean.csv")

print("\n=== CSV 与 Parquet 对比 ===")

# CSV
t0 = time.time()
pd.read_csv(csv_path)
csv_time = time.time() - t0
csv_size = os.path.getsize(csv_path) / 1024

# Parquet
t0 = time.time()
pd.read_parquet(parquet_output)
parquet_time = time.time() - t0
parquet_size = os.path.getsize(parquet_output) / 1024

print(f"CSV    读取耗时: {csv_time:.3f}s  文件大小: {csv_size:.1f} KB")
print(f"Parquet 读取耗时: {parquet_time:.3f}s  文件大小: {parquet_size:.1f} KB")
print(f"\n速度提升: {csv_time/parquet_time:.1f}x")
print(f"体积压缩: {csv_size/parquet_size:.1f}x")


=== CSV 与 Parquet 对比 ===
CSV    读取耗时: 0.048s  文件大小: 1972.4 KB
Parquet 读取耗时: 0.010s  文件大小: 871.9 KB

速度提升: 4.8x
体积压缩: 2.3x


### Parquet 与 CSV 对比分析

**在本次数据规模下**：
- 由于数据量相对较小（10只股票5年数据），两种格式的速度和体积差异可能不太明显
- Parquet 的压缩优势在数据量增大时会更加显著

**差异更显著的场景**：
- **大数据量**：当数据量达到 GB 级别时，Parquet 的压缩优势和读取速度优势会非常明显
- **列式查询**：当只需要读取部分列时，Parquet 可以只加载需要的列，而 CSV 必须读取全部
- **类型安全**：Parquet 保存了完整的类型信息，读取时不需要重新推断类型
- **长期存档**：Parquet 是标准化的列式存储格式，适合长期数据存档

## 清洗完成

所有数据清洗和存储任务已完成。